# Cell Type Annotation using Scanpy

This notebook helps annotate the SC clusters using marker genes and visualization.

In [1]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sc.set_figure_params(figsize=(8, 8), dpi=100)
sns.set_style('whitegrid')

print("Libraries loaded successfully!")

Libraries loaded successfully!


## Load clustered data

In [2]:
# Load the clustered SC data
adata = sc.read_h5ad('./stage1_results/sc_adata_clustered.h5ad')

print(f"Loaded data shape: {adata.shape}")
print(f"Clusters: {sorted(adata.obs['leiden'].unique())}")
print(f"\nDataset info:")
print(adata)

Loaded data shape: (18024, 27719)
Clusters: ['0', '1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '3', '30', '31', '32', '33', '34', '35', '4', '5', '6', '7', '8', '9']

Dataset info:
AnnData object with n_obs × n_vars = 18024 × 27719
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mito', 'subtype', 'celltype_subset', 'celltype_minor', 'celltype_major', 'cell_type', 'n_genes', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'sample', 'modality', 'leiden'
    uns: 'log1p', 'rank_genes_groups'


In [4]:
X = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
print(np.min(X), np.max(X))

0.0 9.023737


In [8]:
composition_X = pd.read_csv('./stage2_results/CID4465/CID4465_reconstructed_all_genes.csv', index_col=0)

In [9]:
print(np.min(composition_X), np.max(composition_X))

0.0 5.361783


## Visualize UMAP with clusters

In [ ]:
# Compute UMAP if not already computed
if 'X_umap' not in adata.obsm:
    print("Computing UMAP...")
    sc.tl.umap(adata)
    print("UMAP computed!")
else:
    print("UMAP already exists in the data")

# Plot clusters
sc.pl.umap(adata, color='leiden', legend_loc='on data', title='SC Clusters', show=True)

## Load marker genes

In [ ]:
# Load marker genes list
with open('./stage1_results/marker_genes.txt', 'r') as f:
    marker_genes = [line.strip() for line in f.readlines()]

print(f"Total marker genes: {len(marker_genes)}")
print(f"\nFirst 20 marker genes:")
print(marker_genes[:20])

## Visualize marker gene expression by cluster

In [ ]:
# Select marker genes for visualization (adjust as needed)
# For example, visualize first 10 marker genes
genes_to_plot = marker_genes[:10]

# Check which genes are available
available_genes = [g for g in genes_to_plot if g in adata.var_names]
print(f"Available genes to plot: {available_genes}")

if available_genes:
    sc.pl.umap(adata, color=available_genes[:6], ncols=3, title='Marker Gene Expression', show=True)

## Dot plot: marker genes expression across clusters

In [ ]:
# Create dot plot to visualize marker gene expression patterns
# This helps identify cluster identity

# Select subset of marker genes for clarity (adjust the number as needed)
genes_for_dotplot = marker_genes[:30]
available_genes_dotplot = [g for g in genes_for_dotplot if g in adata.var_names]

if available_genes_dotplot:
    plt.figure(figsize=(14, 8))
    sc.pl.dotplot(adata, var_names=available_genes_dotplot, groupby='leiden', 
                   title='Marker Gene Expression by Cluster', show=True)

## Heatmap: marker genes expression across clusters

In [ ]:
# Create heatmap for better visualization
genes_for_heatmap = marker_genes[:20]
available_genes_heatmap = [g for g in genes_for_heatmap if g in adata.var_names]

if available_genes_heatmap:
    sc.pl.heatmap(adata, var_names=available_genes_heatmap, groupby='leiden',
                   title='Marker Gene Expression Heatmap', show=True)

## Manual cell type annotation

Based on the marker gene expression patterns above, annotate each cluster with a cell type.

In [ ]:
# Define cell type mapping for each cluster
# Modify this dictionary based on your marker gene analysis

cell_type_mapping = {
    # Example mapping - CHANGE THESE BASED ON YOUR MARKER GENES!
    # '0': 'Cell Type 1',
    # '1': 'Cell Type 2',
    # ...
}

# Print clusters to map
clusters = sorted(adata.obs['leiden'].unique())
print(f"Clusters to annotate: {clusters}")
print(f"\nPlease modify cell_type_mapping above with your annotations")
print(f"\nExample format:")
for cluster in clusters[:3]:
    print(f"    '{cluster}': 'YourCellType',")

In [ ]:
# Convert cluster IDs to strings for mapping
clusters_str = [str(c) for c in adata.obs['leiden'].unique()]

# Create cell type annotation
adata.obs['cell_type'] = adata.obs['leiden'].astype(str).map(cell_type_mapping)

# Check for unmapped clusters
unmapped = adata.obs['cell_type'].isna().sum()
if unmapped > 0:
    print(f"Warning: {unmapped} cells are unmapped (cell_type_mapping incomplete)")
    print(f"Unmapped clusters: {adata.obs[adata.obs['cell_type'].isna()]['leiden'].unique()}")
else:
    print("All clusters have been annotated!")

print(f"\nCell type distribution:")
print(adata.obs['cell_type'].value_counts())

## Visualize annotated clusters

In [ ]:
# Plot UMAP colored by cell type annotation
sc.pl.umap(adata, color='cell_type', legend_loc='on data', title='Annotated Cell Types', show=True)

## Save annotated data

In [ ]:
# Save the annotated data
output_file = './stage1_results/sc_adata_annotated.h5ad'
adata.write_h5ad(output_file)
print(f"Saved annotated data to: {output_file}")

# Also save cell type mapping as CSV
mapping_df = pd.DataFrame([
    {'Cluster': cluster, 'Cell_Type': cell_type} 
    for cluster, cell_type in cell_type_mapping.items()
])

mapping_file = './stage1_results/cell_type_mapping.csv'
mapping_df.to_csv(mapping_file, index=False)
print(f"Saved cell type mapping to: {mapping_file}")

print("\nAnnotation complete!")

## Summary statistics

In [ ]:
print("="*60)
print("ANNOTATION SUMMARY")
print("="*60)
print(f"\nTotal cells: {adata.n_obs}")
print(f"Total genes: {adata.n_vars}")
print(f"Number of clusters: {len(clusters)}")
print(f"\nCell type distribution:")
for cell_type in adata.obs['cell_type'].unique():
    if pd.notna(cell_type):
        count = (adata.obs['cell_type'] == cell_type).sum()
        pct = 100 * count / adata.n_obs
        print(f"   {cell_type}: {count} cells ({pct:.1f}%)")